# Modified GSEA of MAST DEGs against Hallmark Pathways: Remove PSM genes

This notebook contains analysis of bortezomib DEGs after removal of proteasomal (PSM) genes.

The goal of this analysis is to determine which pathway enrichment results are strongly dependent on the set of genes related to the proteasome, and to identify additional pathways that may be enriched when the dominant effects on this set of strongly related genes are removed.

The analysis itself is identical to the generalized notebook for GSEA of MAST DEGs, with the addition of the **DEG Filtering** section, which selects only Bortezomib conditions and removes the set of proteasomal genes described in Mao, 2021:  

Mao, Y. Structure, Dynamics and Function of the 26S Proteasome. in Macromolecular Protein Complexes III: Structure and Function (eds. Harris, J. R. & Marles-Wright, J.) 1–151 (Springer International Publishing, 2021).

## Setup

For this analysis, we'll compare our DEGs to the MSigDB Hallmark Gene Sets, available in the `msidbr` package. We'll need to install this package if it's not already present.

In [1]:
ip <- installed.packages()
if(!"msigdbr" %in% rownames(ip)) {
    install.packages("msigdbr", upgrade = "never")
}

## Load packages

hise: The Human Immune System Explorer R SDK package  
purrr: Functional programming tools  
dplyr: Dataframe handling functions  
fgsea: Fast Gene Set Enrichment Analysis  
msigdbr: MSigDB gene sets

In [2]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(hise)
quiet_library(purrr)
quiet_library(dplyr)
quiet_library(fgsea)
quiet_library(msigdbr)

Warning message:
“package ‘purrr’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘fgsea’ was built under R version 4.4.2”
Warning message:
“package ‘msigdbr’ was built under R version 4.4.3”


## Retrieve files

Now, we'll use the HISE SDK package to retrieve the MAST DEG results file based on its UUID. This will be placed in the `cache/` subdirectory by default.

In [3]:
file_uuid <- list(
    "fc83b89f-fd26-43b8-ac91-29c539703a45"
)

In [4]:
fres <- cacheFiles(file_uuid)

[2026-09-28 16:22:42] INFO  Calling cacheFiles
[1] "downloading fileID fc83b89f-fd26-43b8-ac91-29c539703a45"
[2026-09-28 16:22:48] INFO  Finished cacheFiles (success=TRUE, time_elapsed=3.765s)


In [5]:
fres

[1] "/home/workspace/input/1918706177/cohorts/fc83b89f-fd26-43b8-ac91-29c539703a45/all_mast_deg_2023-09-06.csv"

In [6]:
psm_genes <- read.csv("../common/gene_sets/mao_proteasome_genes.csv")

### Prepare DEG lists

To rank genes, we'll convert nomP to -log10(nomP), and incorporate the direction of differential expression by multiplying by the direction of effect size (sign(logFC)).

In [7]:
all_deg <- read.csv(fres)
all_deg$treat_time_type <- paste0(
    all_deg$fg, "_", 
    all_deg$timepoint, "_", 
    all_deg$aifi_cell_type)

Prior to ranking, we'll need to resolve missing `logFC` values. These can occur if one of the groups used for DEG analysis had no expression of the gene.

In [8]:
all_deg %>%
  filter(is.na(logFC)) %>%
  head()

,aifi_cell_type,timepoint,fg,bg,n_sample,gene,coef_C,coef_D,logFC,nomP,adjP,treat_time_type
,<chr>,<int>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,t_cd4_em,4,bortezomib,dmso,180,TFDP1,NA,3.196997,NA,3.817725e-05,0.1481468,bortezomib_4_t_cd4_em
2,t_cd4_treg,4,bortezomib,dmso,78,ABCA3,NA,-2.453613,NA,1.337154e-02,0.9999222,bortezomib_4_t_cd4_treg
3,t_cd4_treg,4,bortezomib,dmso,78,AC005070.3,NA,-2.259963,NA,3.110707e-02,0.9999222,bortezomib_4_t_cd4_treg
4,t_cd4_treg,4,bortezomib,dmso,78,AC006504.5,NA,-2.470421,NA,1.243915e-02,0.9999222,bortezomib_4_t_cd4_treg
5,t_cd4_treg,4,bortezomib,dmso,78,AC007686.3,NA,-2.265379,NA,3.041017e-02,0.9999222,bortezomib_4_t_cd4_treg
6,t_cd4_treg,4,bortezomib,dmso,78,AC010754.1,NA,-2.272969,NA,2.952072e-02,0.9999222,bortezomib_4_t_cd4_treg


When this occurs, we can use the sign of `coef_D` to determine the direction of expression change, rather than using the missing `logFC` value.

In [9]:
all_deg <- all_deg %>%
  mutate(direction = ifelse(
      is.na(logFC),
      sign(coef_D), # if missing logFC, use coef_D
      sign(logFC) # otherwise, use logFC
  ))

We also need to avoid nomP values of 0. These will cause NA values due to log transformation. We'll convert these to `1e-300` so that they have a non-zero value.

In [10]:
all_deg <- all_deg %>%
  mutate(nomP = ifelse(
      nomP == 0,
      1e-300, # if zero, change to 1e-300
      nomP # otherwise, keep the value
  ))

## DEG Filtering

Filter to select bortezomib conditions

In [11]:
all_deg <- all_deg %>%
  filter(fg == "bortezomib")

Filter to remove proteasomal genes

In [12]:
all_deg <- all_deg %>%
  filter(!gene %in% psm_genes$gene)

## Rank and Split for analysis

In [13]:
deg_list <- split(all_deg, all_deg$treat_time_type)

In [14]:
deg_list <- map(
    deg_list,
    function(deg) {
        deg %>%
          mutate(rank_val = -log10(nomP) * direction) %>%
          arrange(desc(rank_val))
    }
)

In [15]:
rank_list <- map(
    deg_list,
    function(deg) {
        v <- deg$rank_val
        names(v) <- deg$gene
        v
    }
)

## Load Gene Sets

For use with `fastgsea`, we'll use the Reactome gene sets that were previously built in `02-R_mast-deg_reactome_gsea.ipynb`.

In [16]:
reactome_gene_sets_uuid <- "f73aae8d-8984-4c31-a0d2-ae167353b40f"
reactome_gene_sets_rds <- cacheFiles(list(reactome_gene_sets_uuid))
reactome <- readRDS(reactome_gene_sets_rds)

[2026-09-28 16:22:55] INFO  Calling cacheFiles
[1] "downloading fileID f73aae8d-8984-4c31-a0d2-ae167353b40f"
[2026-09-28 16:23:01] INFO  Finished cacheFiles (success=TRUE, time_elapsed=3.496s)


In [17]:
names(reactome)

[1] "root_tb"  "l1_tb"    "l2_tb"    "l3_tb"    "l4_tb"    "set_list"

In [18]:
reactome_df <- data.frame(
    pathway = names(reactome$set_list),
    n_pathway_genes = map_int(reactome$set_list, length),
    pathway_genes = map_chr(reactome$set_list, paste, collapse = ";")
)
reactome_df <- unique(reactome_df)

## Get Pathways table
This table from Reactome gives us the names of each pathway.

V87 of the Reactome database was used for our previous analysis (2673 human pathways)

In [19]:
# Pathway gene sets
download.file(
    "https://download.reactome.org/87/ReactomePathways.txt", 
    "ReactomePathways.txt"
)

In [20]:
pw <- readLines("ReactomePathways.txt")
pw <- strsplit(pw, split = "\t")

In [21]:
pw <- map(pw, as.list)
pw <- map(pw, function(l) { names(l) <- c("id", "name", "species"); l })
pw <- map_dfr(pw, as.data.frame)

In [22]:
head(pw)

,id,name,species
,<chr>,<chr>,<chr>
1,R-BTA-73843,5-Phosphoribose 1-diphosphate biosynthesis,Bos taurus
2,R-BTA-1971475,A tetrasaccharide linker sequence is required for GAG synthesis,Bos taurus
3,R-BTA-1369062,ABC transporters in lipid homeostasis,Bos taurus
4,R-BTA-382556,ABC-family proteins mediated transport,Bos taurus
5,R-BTA-9033807,ABO blood group biosynthesis,Bos taurus
6,R-BTA-418592,ADP signalling through P2Y purinoceptor 1,Bos taurus


In [23]:
pw <- pw %>%
  filter(species == "Homo sapiens") %>%
  select(id, name)

In [24]:
nrow(pw)

[1] 2673

## Run GSEA

In [25]:
parallel_param <- BiocParallel::MulticoreParam(
    workers = 4, 
    progressbar = FALSE
)

In [26]:
fgsea_res <- map(
    rank_list,
    function(ranks) {
        fgsea(
            pathways = reactome$set_list,
            stats    = ranks,
            minSize  = 10,
            maxSize  = 500,
            BPPARAM  = parallel_param
        )
    }
)

Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.04% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


### Format results

In [27]:
deg_meta <- map(
    deg_list,
    function(deg) {
        list(
            fg = deg$fg[1],
            bg = deg$bg[1],
            timepoint = deg$timepoint[1],
            aifi_cell_type = deg$aifi_cell_type[1]
        )
    }
)

In [28]:
head(fgsea_res[[1]])

pathway,pval,padj,log2err,ES,NES,size,leadingEdge
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<list>
R-HSA-1059683,7.523992e-01,0.8661522776,0.05688642,0.3765772,0.7363174,10,PTPN11
R-HSA-109581,2.123793e-02,0.0808198080,0.35248786,0.5712000,1.5114234,90,"UBC , NMT1 , TFDP2 , SEM1 , UBB , LY96 , BAX , PPP3CC, MAPK8 , YWHAG , TICAM1, LMNA , BID , ADD1 , YWHAZ , DNM1L , PTK2 , CTNNB1"
R-HSA-109606,2.455285e-01,0.4513662204,0.10839426,0.5265877,1.2261959,40,"NMT1 , TFDP2 , BAX , PPP3CC, MAPK8 , YWHAG , BID , YWHAZ"
R-HSA-110312,7.361119e-03,0.0397744442,0.40701792,0.8676164,1.7758298,13,"UBC, UBB"
R-HSA-110313,1.984204e-06,0.0003881104,0.62725674,0.9004897,2.0166620,31,"VCP , UBC , NPLOC4, UBB"
R-HSA-110314,5.770440e-03,0.0344115256,0.40701792,0.7779409,1.6902027,26,"UBC, UBB"


In [29]:
formatted_fgsea_res <- map2_dfr(
    fgsea_res,
    deg_meta,
    function(res, meta) {
        res %>%
          mutate(
              leadingEdge = map_chr(leadingEdge, paste, collapse = ";"),
              fg = meta$fg,
              bg = meta$bg,
              timepoint = meta$timepoint,
              aifi_cell_type = meta$aifi_cell_type
          ) %>%
          left_join(reactome_df, by = "pathway") %>%
          rename(id = pathway,
                 nomP = pval,
                 adjP = padj,
                 n_leadingEdge = size) %>%
          left_join(pw, by = "id") %>%
          rename(pathway = name) %>%
          select(fg, bg, timepoint, aifi_cell_type,
                 id, pathway, NES, nomP, adjP, 
                 n_leadingEdge, n_pathway_genes,
                 leadingEdge, pathway_genes) %>%
          arrange(desc(NES))

    }
)

In [30]:
head(formatted_fgsea_res)

fg,bg,timepoint,aifi_cell_type,id,pathway,NES,nomP,adjP,n_leadingEdge,n_pathway_genes,leadingEdge,pathway_genes
<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<int>,<chr>,<chr>
bortezomib,dmso,24,t_cd4_cm,R-HSA-9755511,KEAP1-NFE2L2 pathway,2.108072,3.347253e-08,1.636807e-05,54,132,VCP;UBC;NPLOC4;TXNRD1;TXN;SQSTM1;TALDO1;MAFG;GSR;PRDX1;BACH1;GCLM;SEM1;UBB;NFE2L2;SP1;KEAP1;PGD;CUL1;GCLC,ABCC1;ABCC3;ABCF2;ABCG2;AKT1;AKT2;AKT3;AMER1;AREG;ATF4;BACH1;BCL2;BCL2L1;BRCA1;BTRC;CCL2;CDKN1A;CDKN2A;CREBBP;CSNK2A1;CSNK2A2;CSNK2B;CUL1;CUL3;DPP3;EGF;EIF2AK3;EP300;FBXL17;G6PD;GCLC;GCLM;GSK3B;GSR;GSTA1;GSTA3;HMOX1;IDH1;IL8;KEAP1;MAFG;MAFK;MAP1LC3B;ME1;MIR155;MIR196A1;MIR98;MIRLET7B;MIRLET7C;MUL1;MYC;NFE2L2;NFKB1;NOTCH1;NPLOC4;NQO1;PALB2;PDGFA;PGD;PRDX1;PRKAA2;PRKCD;PRKCI;PSMA1;PSMA2;PSMA3;PSMA4;PSMA5;PSMA6;PSMA7;PSMA8;PSMB1;PSMB10;PSMB11;PSMB2;PSMB3;PSMB4;PSMB5;PSMB6;PSMB7;PSMB8;PSMB9;PSMC1;PSMC2;PSMC3;PSMC4;PSMC5;PSMC6;PSMD1;PSMD10;PSMD11;PSMD12;PSMD13;PSMD14;PSMD2;PSMD3;PSMD4;PSMD5;PSMD6;PSMD7;PSMD8;PSMD9;PSME1;PSME2;PSME3;PSME4;PSMF1;RBX1;RELA;RPS27A;SEM1;SESN1;SESN2;SKP1;SKP2;SLC7A11;SOD3;SP1;SQSTM1;SRXN1;TALDO1;TKT;TRIM21;TXN;TXNRD1;UBA52;UBB;UBC;UFD1L;VCP;WIPF1;X
bortezomib,dmso,24,t_cd4_cm,R-HSA-9711123,Cellular response to chemical stress,2.055989,8.966422e-08,2.923054e-05,115,219,VCP;UBC;NPLOC4;TXNRD1;TXN;SQSTM1;TALDO1;MAFG;GSR;PRDX1;BACH1;GCLM;SEM1;SOD1;GSTP1;UBB;NFE2L2;SP1;NDUFA4;KEAP1;PGD;CUL1;GCLC;BLVRB,ABCC1;ABCC3;ABCF2;ABCG2;AKT1;AKT2;AKT3;ALB;AMER1;AQP8;AREG;ATF4;ATOX1;ATP7A;BACH1;BCL2;BCL2L1;BLVRA;BLVRB;BRCA1;BTRC;CARM1;CAT;CCL2;CCS;CDKN1A;CDKN2A;CHD9;COX11;COX14;COX16;COX18;COX19;COX20;COX4I1;COX5A;COX5B;COX6A1;COX6B1;COX6C;COX7A2L;COX7B;COX7C;COX8A;CREBBP;CSNK2A1;CSNK2A2;CSNK2B;CUL1;CUL3;CYBA;CYBB;CYCS;DPP3;EGF;EIF2AK3;EP300;ERO1A;FABP1;FBXL17;G6PD;GCLC;GCLM;GPX1;GPX2;GPX3;GPX5;GPX6;GPX7;GPX8;GSK3B;GSR;GSTA1;GSTA3;GSTP1;HBA1;HBB;HDAC3;HELZ2;HM13;HMOX1;HMOX2;IDH1;IL8;KEAP1;LRPPRC;MAFG;MAFK;MAP1LC3B;ME1;MED1;MIR155;MIR196A1;MIR98;MIRLET7B;MIRLET7C;MT-CO1;MT-CO2;MT-CO3;MUL1;MYC;NCF1;NCF2;NCF4;NCOA1;NCOA2;NCOA6;NCOR1;NCOR2;NDUFA4;NFE2L2;NFKB1;NLRP3;NOTCH1;NOX4;NOX5;NPLOC4;NQO1;NUDT2;P4HB;PALB2;PDGFA;PGD;PPARA;PRDX1;PRDX2;PRDX3;PRDX6;PRKAA2;PRKCD;PRKCI;PSMA1;PSMA2;PSMA3;PSMA4;PSMA5;PSMA6;PSMA7;PSMA8;PSMB1;PSMB10;PSMB11;PSMB2;PSMB3;PSMB4;PSMB5;PSMB6;PSMB7;PSMB8;PSMB9;PSMC1;PSMC2;PSMC3;PSMC4;PSMC5;PSMC6;PSMD1;PSMD10;PSMD11;PSMD12;PSMD13;PSMD14;PSMD2;PSMD3;PSMD4;PSMD5;PSMD6;PSMD7;PSMD8;PSMD9;PSME1;PSME2;PSME3;PSME4;PSMF1;PTK6;RBX1;RELA;RPS27A;RXRA;SCO1;SCO2;SEM1;SESN1;SESN2;SIN3A;SIN3B;SKP1;SKP2;SLC7A11;SMARCD3;SOD1;SOD2;SOD3;SP1;SQSTM1;SRXN1;STAP2;STAT3;SURF1;TACO1;TALDO1;TBL1X;TBL1XR1;TGS1;TKT;TRIM21;TXN;TXN2;TXNIP;TXNRD1;TXNRD2;UBA52;UBB;UBC;UFD1L;VCP;WIPF1;X
bortezomib,dmso,24,t_cd4_cm,R-HSA-73893,DNA Damage Bypass,2.026996,1.981847e-06,3.881104e-04,39,48,VCP;UBC;NPLOC4;UBB,CUL4A;CUL4B;DDB1;DTL;ISG15;MAD2L2;NPLOC4;PCLAF;PCNA;POLD1;POLD2;POLD3;POLD4;POLE;POLE2;POLE3;POLE4;POLH;POLI;POLK;RAD18;RBX1;RCHY1;REV1;REV3L;RFC1;RFC2;RFC3;RFC4;RFC5;RPA1;RPA2;RPA3;RPS27A;SPRTN;TRIM25;UBA52;UBA7;UBB;UBC;UBE2B;UBE2L6;UFD1L;USP1;USP10;USP43;VCP;WDR48
bortezomib,dmso,24,t_cd4_cm,R-HSA-110313,Translesion synthesis by Y family DNA polymerases bypasses lesions on DNA template,2.016662,1.984204e-06,3.881104e-04,31,39,VCP;UBC;NPLOC4;UBB,ISG15;MAD2L2;NPLOC4;PCLAF;PCNA;POLD1;POLD2;POLD3;POLD4;POLE;POLE2;POLE3;POLE4;POLH;POLI;POLK;RCHY1;REV1;REV3L;RFC1;RFC2;RFC3;RFC4;RFC5;RPA1;RPA2;RPA3;RPS27A;SPRTN;TRIM25;UBA52;UBA7;UBB;UBC;UBE2L6;UFD1L;USP10;USP43;VCP
bortezomib,dmso,24,t_cd4_cm,R-HSA-9759194,Nuclear events mediated by NFE2L2,2.004313,3.791104e-06,4.634625e-04,36,97,UBC;TXNRD1;TXN;SQSTM1;TALDO1;MAFG;GSR;PRDX1;BACH1;GCLM;SEM1;UBB;NFE2L2;SP1;KEAP1;PGD;CUL1;GCLC,ABCC1;ABCC3;ABCF2;ABCG2;AREG;ATF4;BACH1;BCL2;BCL2L1;BTRC;CCL2;CDKN2A;CREBBP;CUL1;EGF;EP300;G6PD;GCLC;GCLM;GSK3B;GSR;GSTA1;GSTA3;HMOX1;IDH1;IL8;KEAP1;MAFG;MAFK;ME1;MYC;NFE2L2;NFKB1;NOTCH1;NQO1;PDGFA;PGD;PRDX1;PRKAA2;PSMA1;PSMA2;PSMA3;PSMA4;PSMA5;PSMA6;PSMA7;PSMB1;PSMB10;PSMB2;PSMB3;PS

## Write output file

Write the metadata as a .csv for later use. We remove `row.names` and set `quote = FALSE` to simplify the outputs and increase compatibility with other tools.

In [31]:
dir.create("output")

In [32]:
out_file <- paste0("output/bortezomib_no-PSM_reactome_gsea_res_", Sys.Date(), ".csv")
write.csv(
    formatted_fgsea_res,
    out_file,
    row.names = FALSE,
    quote = FALSE
)

## Store results in HISE

Finally, we store the output file in our Collaboration Space for later retrieval and use. We need to provide the UUID for our Collaboration Space (aka `studySpaceId`), as well as a title for this step in our analysis process.

The hise function `uploadFiles()` also requires the FileIDs from the original fileset for reference, which we used above when the DEG results were retrieved (`file_uuid`)

In [33]:
study_space_uuid <- "40df6403-29f0-4b45-ab7d-f46d420c422e"
title <- paste("VRd TEA-seq Reactome Bor No-PSM GSEA Analysis", Sys.Date())

In [34]:
out_list <- as.list(out_file)

In [35]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = list(file_uuid, reactome_gene_sets_uuid),
    store = "project",
    doPrompt = FALSE
)

[2026-09-28 16:23:39] INFO  Calling uploadFiles
[2026-09-28 16:24:21] INFO  Finished uploadFiles (success=TRUE, time_elapsed=40.381s)


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "37b5f553-960b-4925-891d-7a30a08abf47"

$ProcessId
[1] "6703899a-e841-40d7-ae3b-4ab8b8bf4ff9"

$WorkflowId
[1] "43d00734-83f7-4c44-9e89-bd800eeaa67f"

$FileIds
$FileIds[[1]]
[1] "97876b0a-3354-4fc1-ab49-81fd22bf362f"

In [36]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/archrpixiv11/.pixi/envs/default/lib/libopenblasp-r0.3.32.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] msigdbr_26.1.1 fgsea_1.32.2   dplyr_1.2.1    purrr_1.2.1    hise_2.16.0   

loaded via a namespace (and not attached):
 [1] generics_0.1.4      bitops_1.0-9        stringi_1.8.7      
 [4] lattice_0.22-9      digest_0.6.39       magrittr_2.0.5     
 [7] evaluate_1.0.5      grid_4.4.1         